# YOLOv8s: 단일 클래스("tile") 위치 탐지 전용 학습

`train_models.ipynb`의 YOLO는 53클래스(색+숫자 조합)를 직접 맞추려고 했는데, 이제 색/숫자는
별도 ResNet18 분류기 2개가 담당하므로 YOLO에게 그 부담을 지울 필요가 없습니다. 이 노트북은
**"타일이 있다/없다"(1클래스) 위치 탐지만** 학습시켜서, loss가 localization(박스 정확도)에
온전히 집중되게 합니다 — recall/박스 품질이 오르면 뒤에 붙는 crop+분류기 파이프라인
(`infer_test_images.ipynb`)의 전체 정확도도 같이 올라갑니다.

**모델**: 처음엔 `yolo26n`으로 시도했는데 1클래스로 단순화해도 전혀 감지를 못 하는 문제가 있어,
검증된 `YOLOv8s`로 교체했습니다.

**아예 감지를 못 하던 진짜 원인**: 처음엔 이미지를 symlink로만 연결하고 `Path.resolve()`로
경로를 만들었는데, `resolve()`가 symlink를 원본 경로로 따라가버려서 Ultralytics가 라벨을 찾을 때
새로 만든 1클래스 라벨이 아니라 `rummikub_data_3`의 원본 53클래스 라벨을 읽고 있었습니다.
`dataset.yaml`엔 클래스가 1개뿐인데 라벨엔 0~52가 섞여 있었으니 학습이 사실상 정상적으로 되지
않았을 가능성이 큽니다. 지금은 symlink 대신 **실제 복사**로 바꿔서 이 문제를 없앴고, 라벨이 전부
class_id `0`인지 검사하는 안전장치 셀도 추가했습니다.

## 이 노트북이 하는 일
1. `rummikub_data_3`를 건드리지 않고, 이미지를 **복사** + 라벨만 전부 class_id `0`(`tile`)로
   바꾼 새 데이터셋(`rummikub_data_3_tile_only`)을 만듭니다.
2. 만들어진 라벨이 전부 class_id `0`인지 전수 검사합니다.
3. `train_models.ipynb`와 동일하게 train의 10%를 val로 떼어 학습하고, `test`는 그대로 둡니다.
4. YOLOv8s를 이 1클래스 데이터로 학습하고 `weights/yolov8s_tile_detector_best.pt`로 저장합니다.

cwd가 `.../boardgame-ai/rummikub_0730`라고 가정한 상대경로만 쓰며, 로컬 프로젝트 경로 의존은
없습니다. 아직 실행하지 않은 상태입니다.

In [ ]:
from __future__ import annotations

import random
import shutil
from pathlib import Path

import torch

SRC_ROOT = Path("rummikub_data_3")
TILE_ROOT = Path("rummikub_data_3_tile_only")
WEIGHTS_OUT = Path("weights")
WEIGHTS_OUT.mkdir(exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## 1. 단일 클래스(`tile`) 데이터셋 만들기 (원본 비파괴)

이미지는 그대로 복사하고, 라벨의 class_id만 전부 `0`으로 바꿔서 새로 씁니다.
`rummikub_data_3`의 원본 53클래스 라벨은 그대로 보존됩니다.

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}


def build_tile_only_dataset(src_root: Path, dst_root: Path) -> dict:
    stats = {"images": 0, "boxes": 0}
    for split in ("train", "test"):
        src_image_dir = src_root / "images" / split
        src_label_dir = src_root / "labels" / split
        if not src_image_dir.is_dir():
            continue

        dst_image_dir = dst_root / "images" / split
        dst_label_dir = dst_root / "labels" / split
        dst_image_dir.mkdir(parents=True, exist_ok=True)
        dst_label_dir.mkdir(parents=True, exist_ok=True)

        for image_path in sorted(src_image_dir.iterdir()):
            if image_path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            label_path = src_label_dir / f"{image_path.stem}.txt"
            if not label_path.is_file():
                continue

            dst_image_path = dst_image_dir / image_path.name
            if not dst_image_path.exists():
                shutil.copy2(image_path, dst_image_path)

            lines = [
                line for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()
            ]
            remapped = []
            for line in lines:
                _, cx, cy, w, h = line.split()[:5]
                remapped.append(f"0 {cx} {cy} {w} {h}")
            (dst_label_dir / f"{image_path.stem}.txt").write_text(
                "\n".join(remapped) + ("\n" if remapped else ""), encoding="utf-8"
            )
            stats["images"] += 1
            stats["boxes"] += len(remapped)
    return stats


build_stats = build_tile_only_dataset(SRC_ROOT, TILE_ROOT)
print(f"tile-only 데이터셋 생성 완료: 이미지 {build_stats['images']}장, 박스 {build_stats['boxes']}개")
print("위치:", TILE_ROOT.resolve())

In [ ]:
# 안전장치: 실제로 만들어진 라벨에 0이 아닌 class_id가 섞여 있지 않은지 전수 검사
# (symlink+resolve() 조합에서 원본 53클래스 라벨을 잘못 가리키던 버그가 있었어서,
# 같은 문제가 재발하면 학습 들어가기 전에 바로 잡아내기 위한 체크입니다.)
bad_files = []
for split in ("train", "test"):
    label_dir = TILE_ROOT / "labels" / split
    if not label_dir.is_dir():
        continue
    for label_path in label_dir.glob("*.txt"):
        for line in label_path.read_text(encoding="utf-8").splitlines():
            if line.strip() and not line.startswith("0 "):
                bad_files.append(str(label_path))
                break

if bad_files:
    raise RuntimeError(
        f"class_id가 0이 아닌 라벨 파일이 {len(bad_files)}개 있습니다 (예: {bad_files[:5]}). "
        "build_tile_only_dataset()이 원본 라벨을 잘못 참조하고 있을 가능성이 있습니다."
    )
print(f"검사 완료: {TILE_ROOT} 아래 모든 라벨이 class_id 0만 사용합니다.")

## 2. train/val 분리 + dataset.yaml

In [ ]:
train_image_dir = TILE_ROOT / "images" / "train"
all_train_images = sorted(
    p for p in train_image_dir.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES
)
assert all_train_images, f"이미지가 없습니다: {train_image_dir.resolve()}"

shuffled = all_train_images.copy()
random.Random(RANDOM_SEED).shuffle(shuffled)
VAL_RATIO = 0.1
n_val = max(1, int(len(shuffled) * VAL_RATIO))
val_subset = shuffled[:n_val]
train_subset = shuffled[n_val:]
print(f"train {len(train_subset)}장 / val(train에서 분리) {len(val_subset)}장")

yolo_dir = Path("yolo_dataset_tile_only")
yolo_dir.mkdir(exist_ok=True)
(yolo_dir / "train.txt").write_text("\n".join(str(p.resolve()) for p in train_subset))
(yolo_dir / "val.txt").write_text("\n".join(str(p.resolve()) for p in val_subset))

dataset_yaml_text = (
    f"train: {(yolo_dir / 'train.txt').resolve()}\n"
    f"val: {(yolo_dir / 'val.txt').resolve()}\n"
    f"test: {(TILE_ROOT / 'images' / 'test').resolve()}\n"
    "\n"
    "names:\n"
    "  0: tile\n"
)
dataset_yaml_path = yolo_dir / "dataset.yaml"
dataset_yaml_path.write_text(dataset_yaml_text, encoding="utf-8")
print(dataset_yaml_text)

## 3. YOLOv8s 학습 (1클래스)

In [ ]:
from ultralytics import YOLO

# yolo26n으로는 1클래스(tile) 학습에서도 전혀 감지가 안 되는 문제가 있어
# 검증된 YOLOv8s로 교체했습니다.
YOLO_MODEL_NAME = "yolov8s.pt"
YOLO_EPOCHS = 100
YOLO_IMGSZ = 960

yolo_model = YOLO(YOLO_MODEL_NAME)
yolo_model.train(
    data=str(dataset_yaml_path.resolve()),
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMGSZ,
    batch=-1,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=8,
    patience=25,
    project="runs/detect",
    name="rummikub_yolov8s_tile_only",
    pretrained=True,
    seed=RANDOM_SEED,
    deterministic=True,
    close_mosaic=10,
    plots=True,
)

yolo_best_path = Path(yolo_model.trainer.best)
yolo_out_path = WEIGHTS_OUT / "yolov8s_tile_detector_best.pt"
shutil.copy2(yolo_best_path, yolo_out_path)
print("YOLO(tile-only) 가중치 저장:", yolo_out_path.resolve())

## 참고

- `evaluate_models.ipynb`나 `infer_test_images.ipynb`에서 이 가중치를 쓰려면
  `yolo26_detector_best.pt` 대신 `yolov8s_tile_detector_best.pt`를 가리키도록 경로만 바꾸면 됩니다
  (두 노트북 다 YOLO의 클래스 이름은 참고용으로만 쓰고 최종 라벨은 ResNet 분류기 결과를 쓰므로
  다른 아키텍처/1클래스 모델로 바꿔도 그대로 동작합니다).
- `evaluate_models.ipynb`로 이 모델을 평가할 때는 `rummikub_data_3` 대신
  `rummikub_data_3_tile_only`를 가리키도록 경로를 바꿔야 mAP/precision/recall이 1클래스 기준으로
  올바르게 계산됩니다 (53클래스 라벨과 비교하면 클래스 불일치로 잘못된 점수가 나옵니다).